In [0]:
import requests
import os
from pyspark.sql import Row
from delta.tables import DeltaTable

API_KEY = os.environ.get("ALPHA_VANTAGE_KEY")
SYMBOL = ['GOOGL', 'AAPL', 'MSFT', 'NVDA', 'META', 'TSLA', 'IBM', 'ADBE', 'SSNLF', 'ORCL']
for SYM in (SYMBOL):
    LINK = f'https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY_ADJUSTED&symbol={SYM}&apikey={API_KEY}'
    r = requests.get(LINK)
    data = r.json()
    stock_info = data.get("Weekly Adjusted Time Series", {})
    rows = []
    for key, item in stock_info.items():
        row = Row(
            date=key,
            open=float(item["1. open"]),
            high=float(item["2. high"]),
            low=float(item["3. low"]),
            close=float(item["4. close"]),
            adjusted_close=float(item["5. adjusted close"]),
            volume=int(item["6. volume"]),
            dividend_amount=float(item["7. dividend amount"])
        )
        rows.append(row)
    df = spark.createDataFrame(rows)

    if spark.catalog.tableExists(f"stocks_{SYM}"):
        delta_table = DeltaTable.forName(spark, f"stocks_{SYM}")
        delta_table.alias('existing_table').merge(df.alias('new_table'), "existing_table.date = new_table.date").whenNotMatchedInsertAll().execute()
    else:
        df.write.format("delta").mode("append").saveAsTable(f"stocks_{SYM}")